# Elevation & terrain — SRTM 30 m

NASA SRTM Global 1 arc-second DEM over the Giza plateau — a single static `ee.Image`, so the pipeline composites the one image and writes it as a GeoTIFF.

## Setup

First the imports and the per-notebook output directory. `pyramids` provides `Dataset` (reading + plotting); `earthlens` provides the unified `EarthLens` entry point and the `gee` `Catalog`.

In [ ]:
import os
from pathlib import Path

from pyramids.dataset import Dataset as PyramidsDataset

from earthlens.core import EarthLens
from earthlens.gee import Catalog

OUT_DIR = Path('out') / 'elevation-terrain'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'output directory: {OUT_DIR.resolve()}')

### Credentials

The notebook reads the GEE service-account credentials from the `GEE_SERVICE_ACCOUNT` / `GEE_SERVICE_KEY` environment variables. Both must be set before running this cell.

In [ ]:
SERVICE_ACCOUNT = os.environ['GEE_SERVICE_ACCOUNT']
SERVICE_KEY = os.environ['GEE_SERVICE_KEY']

## Inspect the catalog entry

Before downloading anything, look at what the bundled catalog knows about the asset — bands, cadence, license, provider.

In [ ]:
cat = Catalog()
ds = cat.get_dataset('USGS/SRTMGL1_003')
print(f'title:               {ds.title}')
print(f'ee_type:             {ds.ee_type}')
print(f'spatial_resolution:  {ds.spatial_resolution} m')
print(f'extent.start_date:   {ds.extent.start_date}')
print(f'extent.end_date:     {ds.extent.end_date}')
print(f'default_reducer:     {ds.default_reducer}')
print(f'license:             {ds.license}')
print(f'provider:            {ds.provider}')
print(f'#bands:              {len(ds.bands)}')
print(f'band ids (first 5):  {list(ds.bands)[:5]}')

## Download

Tiny AOI (`[29.9, 30.1]` lat, `[31.1, 31.3]` lon) at 90.0 m, `raw` cadence — keeps the synchronous download under EE's 32768-px per-axis cap. We build the request first, then authenticate and download as separate steps.

In [ ]:
gee = EarthLens(
    data_source="gee",
    start='2000-02-11',
    end='2000-02-12',
    dataset='USGS/SRTMGL1_003',
    variables=['elevation'],
    aoi=[31.1, 29.9, 31.3, 30.1],
    cadence='raw',
    path=str(OUT_DIR),
    scale=90.0,
    reducer='mean',
)
gee.authenticate(service_account=SERVICE_ACCOUNT, service_key=SERVICE_KEY)

`download()` writes the composited image to disk and returns the list of written GeoTIFF paths.

In [ ]:
paths = gee.download(progress_bar=False)
print(f'wrote {len(paths)} GeoTIFF(s):')
for p in paths:
    print(f'  {p}  ({p.stat().st_size / 1024:.1f} KB)')

## Quick preview

Load the first written GeoTIFF through pyramids and render the single band. (`pyramids.dataset.Dataset` is the project's GeoTIFF/NetCDF wrapper.) The dataset's nodata value is masked so the colormap isn't pinned to it.

In [ ]:
# plot() resolves the band and honours its declared no-data, so the array
# is never read, sliced or masked by hand here.
preview = PyramidsDataset.read_file(paths[0])

Render the masked elevation array, and report the value range.

In [ ]:
preview.plot(cmap='viridis', title=f"{'USGS/SRTMGL1_003'} / {'elevation'}")
# stats() reports the band's real range with no-data already excluded,
# which is what the hand-masked nanmin / nanmax calls stood in for.
stats = preview.stats(approx_ok=False)
low, high = stats['min'].iloc[0], stats['max'].iloc[0]
print(f'value range: [{low:.4g}, {high:.4g}]')

## Tracking submitted jobs (asynchronous export)

The download above uses `export_via="url"` — a synchronous `getDownloadURL` round-trip. Nothing was queued, so there's no Earth Engine job to track.

To track an export instead, switch to an asynchronous sink (`drive` / `gcs` / `asset`) and pass `wait_for_export=False` so `.download()` returns a `TaskInfo` at submission time rather than blocking until completion. The cells below submit the same `(asset_id, band, AOI, scale)` request as an `export_via="asset"` task into the service account's own asset folder, then walk the four jobs-API calls (`list_recent_tasks` → `wait_for_task_id` → `ee.data.getAsset` → `ee.data.deleteAsset`) to make the job finish *and* tidy up. See `track-batch-exports.ipynb` for a deeper worked example.

### Prepare the demo asset folder

`GEE._export_via_batch` writes the image at `<asset_id>/<prefix>`, so `asset_id` here is the parent `Folder` asset (not the final image path). Resolve a folder name under the service account's own project.

In [ ]:
import ee

from earthlens.gee import cancel_task, list_recent_tasks, wait_for_task_id

# The asset goes into a `Folder` asset that we own. `GEE._export_via_batch`
# writes the actual image at `<asset_id>/<prefix>`, so `asset_id` here is
# the parent FOLDER (not the final image path). Both must be cleaned up.
_proj = ee.data._get_projects_path().removeprefix('projects/')
DEMO_FOLDER = f'projects/{_proj}/assets/earthlens-demo-elevation-terrain'
print(f'demo folder: {DEMO_FOLDER}')

Best-effort clear any leftover children from a previous run, then (re)create the parent folder — EE requires it to exist before a child write.

In [ ]:
# Best-effort cleanup of leftover children from a previous run (so the
# folder is empty before we try to delete it below).
try:
    for child in ee.data.listAssets({'parent': DEMO_FOLDER}).get('assets', []):
        ee.data.deleteAsset(child['name'])
        print(f'cleared leftover child: {child["name"]}')
    ee.data.deleteAsset(DEMO_FOLDER)
    print(f'cleared leftover folder: {DEMO_FOLDER}')
except Exception:
    pass
# Create the parent folder — EE requires it to exist before a child write.
ee.data.createAsset({'type': 'Folder'}, DEMO_FOLDER)
print(f'created folder: {DEMO_FOLDER}')

### Submit

Same `(asset_id, band, AOI, scale)` request as the sync download above, just routed through `export_via="asset"` + `wait_for_export=False`. Build the request and authenticate as separate steps.

In [ ]:
async_gee = EarthLens(
    data_source="gee",
    start='2000-02-11',
    end='2000-02-12',
    dataset='USGS/SRTMGL1_003',
    variables=['elevation'],
    aoi=[31.1, 29.9, 31.3, 30.1],
    cadence='raw',
    path=str(OUT_DIR),
    scale=90.0,
    reducer='mean',
    export_via='asset',
    asset_id=DEMO_FOLDER,
    wait_for_export=False,
)
async_gee.authenticate(service_account=SERVICE_ACCOUNT, service_key=SERVICE_KEY)

`download()` returns a `TaskInfo` per submitted bucket at the moment the task is queued — no blocking.

In [ ]:
submitted = async_gee.download(progress_bar=False)
task_info = submitted[0]
print(f'submitted: id={task_info.id} state={task_info.state}')
print(f'           description={task_info.description}')

### List + wait

`list_recent_tasks(description_prefix=...)` returns every matching task across the current project; `wait_for_task_id` blocks until the one we care about reaches a terminal state. A real workflow would just poll later from a separate process — the wait here exists so the notebook shows the full success path end-to-end.

In [ ]:
recent = list_recent_tasks(
    description_prefix=task_info.description,
    max_age_min=10,
)
print(f'list_recent_tasks matched {len(recent)} task(s):')
for t in recent:
    print(f'  {t.id}  {t.state:<12} {t.description}')
try:
    final = wait_for_task_id(
        task_info.id,
        poll_seconds=10,
        progress_bar=False,
    )
    print(f'\nfinal state: {final.state}')
except RuntimeError as exc:
    # Raised on FAILED / CANCELLED — cancel-if-still-running
    # so we don't leak an in-flight task on notebook restart.
    print(f'wait_for_task_id raised: {exc}')
    try:
        cancel_task(task_info.id)
    except Exception:
        pass

### Verify + clean up

Confirm the produced asset exists on Earth Engine, then delete it (and the surrounding demo folder) so we don't leak storage between notebook runs. The backend wrote the image at `<DEMO_FOLDER>/<task description>`.

In [ ]:
produced = f'{DEMO_FOLDER}/{task_info.description}'
try:
    meta = ee.data.getAsset(produced)
    print(f'asset exists: type={meta.get("type")} name={meta.get("name")}')
    ee.data.deleteAsset(produced)
    print('asset deleted')
except Exception as exc:
    print(f'verify/delete skipped: {exc}')
# Always try to tear down the parent folder.
try:
    ee.data.deleteAsset(DEMO_FOLDER)
    print(f'folder deleted: {DEMO_FOLDER}')
except Exception as exc:
    print(f'folder delete skipped: {exc}')

## What's on disk

The GeoTIFF is left under the per-notebook `out/` directory for you to inspect. That directory is `.gitignore`d — re-running the notebook overwrites it.

In [ ]:
for p in sorted(OUT_DIR.iterdir()) if OUT_DIR.exists() else []:
    print(f'{p}  ({p.stat().st_size / 1024:.1f} KB)')